# Comparación de modelos — Victimización

Este notebook usa directamente los splits originales encontrados (`X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`) para comparar otros modelos contra el árbol de decisión actual.

La idea es no tocar el notebook original y hacer una comparación limpia con el mismo train/test split.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

RANDOM_STATE = 42


## 1. Cargar los splits originales

In [ ]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv")
y_test = pd.read_csv("y_test.csv")

# Quitar columna índice guardada en CSV
for df in [X_train, X_test, y_train, y_test]:
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)

# Convertir y a Series
y_train = y_train.iloc[:, 0].astype(int)
y_test = y_test.iloc[:, 0].astype(int)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape, y_train.value_counts().to_dict())
print("y_test:", y_test.shape, y_test.value_counts().to_dict())
X_train.head()


## 2. Definir modelos

El árbol actual se reconstruye con `ccp_alpha=0.014954`, que reproduce el árbol podado simple basado en `PC2 <= 0.54`.


In [ ]:
models = {
    "DecisionTree_actual": DecisionTreeClassifier(
        ccp_alpha=0.014954,
        random_state=RANDOM_STATE
    ),
    "LogisticRegression_balanced": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "RandomForest_balanced": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    )
}


## 3. Función de métricas por umbral

In [ ]:
def evaluate_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "threshold": threshold,
        "recall_1": recall_score(y_true, y_pred, zero_division=0),
        "precision_1": precision_score(y_true, y_pred, zero_division=0),
        "specificity_0": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "npv": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
        "f1_1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


## 4. Entrenar y comparar

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

results = []

for model_name, model in models.items():
    print("Entrenando:", model_name)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)

    for threshold in thresholds:
        row = evaluate_threshold(y_test, y_proba, threshold)
        row["model"] = model_name
        row["roc_auc"] = roc_auc
        row["auc_pr"] = auc_pr
        results.append(row)

results_df = pd.DataFrame(results)

# Orden de columnas
cols = [
    "model", "threshold",
    "recall_1", "precision_1", "specificity_0", "npv", "f1_1",
    "balanced_accuracy", "tp", "fp", "tn", "fn", "roc_auc", "auc_pr"
]
results_df = results_df[cols]

results_df.round(4)


## 5. Comparación con umbral estándar 0.50

In [ ]:
standard_050 = (
    results_df[results_df["threshold"] == 0.50]
    .sort_values("balanced_accuracy", ascending=False)
)

standard_050.round(4)


## 6. Modelos con recall de víctima >= 0.90

In [ ]:
high_recall = (
    results_df[results_df["recall_1"] >= 0.90]
    .sort_values(["precision_1", "specificity_0", "balanced_accuracy"], ascending=False)
)

high_recall.round(4).head(30)


## 7. Ver reglas reconstruidas del árbol actual

In [ ]:
dt = models["DecisionTree_actual"]
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(X_train.columns)))


## 8. Guardar resultados

In [ ]:
results_df.to_csv("model_comparison_victim_results.csv", index=False)
standard_050.to_csv("model_comparison_victim_standard_050.csv", index=False)
high_recall.to_csv("model_comparison_victim_high_recall.csv", index=False)

print("Ficheros guardados:")
print("- model_comparison_victim_results.csv")
print("- model_comparison_victim_standard_050.csv")
print("- model_comparison_victim_high_recall.csv")
